# First Run

In [ ]:
import os
import csv
import time
import random
import json
from google import genai
from google.genai import types
from tqdm.auto import tqdm

# --- CONFIGURATION ---
PROMPT_TEMPLATE = """You are an expert manga translator.
Your task is to translate the English text to Thai, using the Japanese text for context.
Analyze the speaker's tone, gender, and the situation from the Japanese text before translating.
Return your response in JSON format with these two keys:
1. "analysis": A brief thought process on tone/context.
2. "thai": The final Thai translation (only the text, no quotes).
Context (Japanese): {jp_text}
Source (English): {en_text}
"""

# Load API keys from local environment, comma-separated
API_KEYS_ENV = os.getenv("GEMINI_API_KEYS", "")
API_KEYS = [key.strip() for key in API_KEYS_ENV.split(",") if key.strip()]
API_KEY_INDEX = 0

def _get_next_api_key():
    global API_KEY_INDEX
    if not API_KEYS:
        return ""
    # Rotate to the next key
    key = API_KEYS[API_KEY_INDEX % len(API_KEYS)]
    API_KEY_INDEX += 1
    return key

def generate(en_text, jp_text):
    # 1. Always grab a fresh key for every request
    api_key = _get_next_api_key()
    
    client = genai.Client(api_key=api_key)
    
    # Use 1.5-flash (Standard, stable model)
    model = "gemini-flash-latest"

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=PROMPT_TEMPLATE.format(jp_text=jp_text, en_text=en_text)),
            ],
        ),
    ]
    
    generate_content_config = types.GenerateContentConfig(
        temperature=1, # Lower temp is better for accuracy
        response_mime_type="application/json", 
        response_schema={
            "type": "OBJECT",
            "properties": {
                "analysis": {"type": "STRING"},
                "thai": {"type": "STRING"}
            },
            "required": ["thai"]
        }
    )
    
    # 2. No Try/Except here. Let errors bubble up to 'call_with_backoff'
    # This ensures if a key fails, we wait and retry (with a new key).
    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=generate_content_config,
    )
    
    # 3. Parse JSON safely
    try:
        json_response = json.loads(response.text)
        return json_response.get("thai", "")
    except Exception:
        # If valid text but bad JSON, just return text
        return response.text

def call_with_backoff(fn, *args, **kwargs):
    delay = 1
    for attempt in range(5):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            msg = str(exc).lower()
            # If we hit a limit, wait longer
            if "429" in msg or "503" in msg or "quota" in msg:
                print(f"Limit hit. Rotating key and sleeping {delay}s...")
                time.sleep(delay + random.uniform(0, 0.5))
                delay = min(delay * 2, 30) # Exponential backoff
                continue
            raise
    return None

# --- MAIN LOOP ---

input_path = "/kaggle/input/practical-project-dataset/jp_to_eng_mapping.csv"
output_path = "/kaggle/working/jp_to_thai_mapping.csv"
save_every = 10

def prepare_output_for_resume(path):
    if not os.path.exists(path):
        return 0, None
    temp_path = path + ".tmp"
    done_count = 0
    with open(path, "r", encoding="utf-8", newline="") as src, open(temp_path, "w", encoding="utf-8", newline="") as dst:
        reader = csv.DictReader(src)
        fieldnames = reader.fieldnames or []
        if "thai" not in fieldnames: fieldnames.append("thai")
        writer = csv.DictWriter(dst, fieldnames=fieldnames)
        writer.writeheader()
        for row in reader:
            if not row.get("thai"): break
            writer.writerow(row)
            done_count += 1
    os.replace(temp_path, path)
    return done_count, fieldnames

def count_data_rows(path):
    with open(path, "r", encoding="utf-8", newline="") as src:
        return max(sum(1 for _ in src) - 1, 0)

done_count, _ = prepare_output_for_resume(output_path)
total_rows = count_data_rows(input_path)
total_remaining = max(total_rows - done_count, 0)

print(f"Starting from row {done_count}. Total remaining: {total_remaining}")

with open(input_path, "r", encoding="utf-8", newline="") as src:
    reader = csv.DictReader(src)
    fieldnames = reader.fieldnames or []
    if "thai" not in fieldnames: fieldnames.append("thai")

    out_exists = os.path.exists(output_path)
    with open(output_path, "a", encoding="utf-8", newline="") as dst:
        writer = csv.DictWriter(dst, fieldnames=fieldnames)
        if not out_exists or os.path.getsize(output_path) == 0:
            writer.writeheader()

        # Skip processed rows
        for _ in range(done_count): next(reader, None)

        # PROGRESS BAR
        for i, row in enumerate(tqdm(reader, total=total_remaining, dynamic_ncols=True), start=done_count):
            jp = row.get("japanese", "")
            eng = row.get("english", "")

            # Call generate
            thai = call_with_backoff(generate, en_text=eng, jp_text=jp)
            
            if thai:
                row["thai"] = thai
                writer.writerow(row)

            # Save periodically
            if (i + 1) % save_every == 0:
                dst.flush()
                os.fsync(dst.fileno())

            # --- RATE LIMIT CONTROL ---
            # 5 RPM per key = 12s per request. 
            # 6 Keys = 2s per request.
            # We sleep 2.1s to be safe.
            time.sleep(2.1)

# Resuming runs

In [ ]:
import os
import csv
import time
import random
import json
import shutil # Added for file copying
from google import genai
from google.genai import types
from tqdm.auto import tqdm

# --- CONFIGURATION ---
PROMPT_TEMPLATE = """You are an expert manga translator.
Your task is to translate the English text to Thai, using the Japanese text for context.
Analyze the speaker's tone, gender, and the situation from the Japanese text before translating.
Return your response in JSON format with these two keys:
1. "analysis": A brief thought process on tone/context.
2. "thai": The final Thai translation (only the text, no quotes).
Context (Japanese): {jp_text}
Source (English): {en_text}
"""

# Load API keys from local environment, comma-separated
API_KEYS_ENV = os.getenv("GEMINI_API_KEYS", "")
API_KEYS = [key.strip() for key in API_KEYS_ENV.split(",") if key.strip()]
API_KEY_INDEX = 0

def _get_next_api_key():
    global API_KEY_INDEX
    if not API_KEYS:
        return ""
    # Rotate to the next key
    key = API_KEYS[API_KEY_INDEX % len(API_KEYS)]
    API_KEY_INDEX += 1
    return key

def generate(en_text, jp_text):
    # 1. Always grab a fresh key for every request
    api_key = _get_next_api_key()
    
    client = genai.Client(api_key=api_key)
    
    model = "gemini-flash-latest"

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=PROMPT_TEMPLATE.format(jp_text=jp_text, en_text=en_text)),
            ],
        ),
    ]
    
    generate_content_config = types.GenerateContentConfig(
        temperature=1, # Lower temp is better for accuracy
        response_mime_type="application/json", 
        response_schema={
            "type": "OBJECT",
            "properties": {
                "analysis": {"type": "STRING"},
                "thai": {"type": "STRING"}
            },
            "required": ["thai"]
        }
    )
    
    # 2. No Try/Except here. Let errors bubble up to 'call_with_backoff'
    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=generate_content_config,
    )
    
    # 3. Parse JSON safely
    try:
        json_response = json.loads(response.text)
        return json_response.get("thai", "")
    except Exception:
        # If valid text but bad JSON, just return text
        return response.text

def call_with_backoff(fn, *args, **kwargs):
    delay = 1
    for attempt in range(5):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            msg = str(exc).lower()
            # If we hit a limit, wait longer
            if "429" in msg or "503" in msg or "quota" in msg:
                # print(f"Limit hit. Rotating key and sleeping {delay}s...")
                time.sleep(delay + random.uniform(0, 0.5))
                delay = min(delay * 2, 30) # Exponential backoff
                continue
            raise
    return None

# --- MAIN LOOP ---

input_path = "/kaggle/input/practical-project-dataset/jp_to_eng_mapping.csv"
output_path = "/kaggle/working/jp_to_thai_mapping.csv"
previous_run_path = "/kaggle/input/practical-project-dataset/jp_to_thai_mapping.csv" 

def load_completed_pairs(file_path, existing_set):
    """Reads a CSV and adds (japanese, english) tuples to the set."""
    if not os.path.exists(file_path):
        return existing_set
    
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            if not reader.fieldnames: return existing_set
            for row in reader:
                jp = row.get("japanese", "").strip()
                en = row.get("english", "").strip()
                th = row.get("thai", "").strip()
                # If we have a translation, mark as done
                if th:
                    existing_set.add((jp, en))
    except Exception:
        pass 
    return existing_set

# 1. SETUP: Copy previous run to output if output doesn't exist yet
if not os.path.exists(output_path) and os.path.exists(previous_run_path):
    print(f"Copying existing results from {previous_run_path}...")
    shutil.copy(previous_run_path, output_path)

# 2. Build the checklist of what is already done
completed_pairs = set()
# We only need to check output_path now, because we either copied the old data there
# or it was already there from a current session.
completed_pairs = load_completed_pairs(output_path, completed_pairs)

print(f"Found {len(completed_pairs)} already translated pairs.")

# 3. Open Files
with open(input_path, "r", encoding="utf-8") as src:
    reader = csv.DictReader(src)
    
    fieldnames = reader.fieldnames or []
    if "thai" not in fieldnames: fieldnames.append("thai")

    # Check if output exists to decide if we need a header
    out_exists = os.path.exists(output_path)
    
    with open(output_path, "a", encoding="utf-8", newline="") as dst:
        writer = csv.DictWriter(dst, fieldnames=fieldnames)
        
        # Only write header if the file is brand new (size 0)
        if not out_exists or os.path.getsize(output_path) == 0:
            writer.writeheader()

        rows = list(reader)
        
        print("Starting translation...")
        for i, row in enumerate(tqdm(rows, dynamic_ncols=True)):
            jp = row.get("japanese", "").strip()
            eng = row.get("english", "").strip()

            # --- CHECK: Is this pair in our completed set? ---
            if (jp, eng) in completed_pairs:
                continue # Skip it!
            # -------------------------------------------------

            # Call generate
            thai = call_with_backoff(generate, en_text=eng, jp_text=jp)
            
            if thai:
                row["thai"] = thai
                writer.writerow(row)
                
                # --- IMMEDIATE FLUSH ---
                dst.flush()
                os.fsync(dst.fileno())
                # -----------------------
                
                # Update set so we don't do duplicates in this same run
                completed_pairs.add((jp, eng))

            # --- RATE LIMIT CONTROL ---
            time.sleep(2.1)